# 3章 LLMベースアプリケーション向け評価ツール - 3-4. LightEval

## 事前準備

当サンプルコードを実行するには、Hugging Face（ https://huggingface.co/ ）のアカウント、およびモデルの利用申請が必要です。

詳細は本書の `3.3.2 MMLU を用いた計量経済学の評価` にあるコラム `訳者補 Llama-3.2-1B-Instruct の利用方法` をご覧ください。



In [ ]:
!pip install lighteval

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of inspect-ai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of inspect-ai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.7/612.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from lighteval.logging.evaluation_tracker import EvaluationTracker
from lighteval.models.transformers.transformers_model import TransformersModelConfig
from lighteval.pipeline import ParallelismManager, Pipeline, PipelineParameters
from lighteval.utils.imports import is_package_available

In [ ]:
def create_evaluation_pipeline(
    output_dir: str,
    model_name: str,
    tasks: str,
    dtype: str = "float16"):
  if is_package_available("accelerate"):
      from datetime import timedelta
      from accelerate import Accelerator, InitProcessGroupKwargs
      accelerator = Accelerator(kwargs_handlers=[InitProcessGroupKwargs(timeout=timedelta(seconds=3000))])
  else:
      accelerator = None
  evaluation_tracker = EvaluationTracker(
      output_dir=output_dir,
      save_details=True,  # サンプルごとの詳細な結果を保存
      push_to_hub=False,  # Hugging Face Hubへ結果を送信する場合はTrue
  )

  #  パイプライン実行パラメータの設定
  pipeline_params = PipelineParameters(
      launcher_type=ParallelismManager.ACCELERATE
  )
  model_config = TransformersModelConfig(
      model_name=model_name,
      dtype=dtype,
      trust_remote_code=True,
      batch_size=8
  )
  # パイプラインの初期化
  pipeline = Pipeline(
      tasks=tasks,
      pipeline_parameters=pipeline_params,
      evaluation_tracker=evaluation_tracker,
      model_config=model_config
  )
  return pipeline


In [ ]:
output_dir="./evals"
model_name="meta-llama/Llama-3.2-1B-Instruct"
tasks="mmlu:econometrics|5"

pipeline = create_evaluation_pipeline(
    output_dir=output_dir,
    model_name=model_name,
    tasks=tasks)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
# 評価の実行
pipeline.evaluate()

Splits: 100%|██████████| 1/1 [00:02<00:00,  2.85s/it]


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# 結果の保存と画面出力
pipeline.save_and_push_results()
pipeline.show_results()

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

|       Task        |Version|Metric|Value|   |Stderr|
|-------------------|-------|------|----:|---|-----:|
|all                |       |em    |0.307|±  |0.0434|
|mmlu:econometrics:5|       |em    |0.307|±  |0.0434|



In [ ]:
results_dict = pipeline.get_results()
import pandas as pd
# DataFrame用にデータを整形
table_data = []
for task_name, metrics in results_dict["results"].items():
    row = {"Task": task_name}
    for metric_name, value in metrics.items():
        # 4桁で丸める
        if isinstance(value, float):
            row[metric_name] = round(value, 4)
        else:
            row[metric_name] = value
    table_data.append(row)

df = pd.DataFrame(table_data)
display(df)

,Task,em,em_stderr
0,mmlu:econometrics:5,0.307,0.0434
1,all,0.307,0.0434
